In [1]:
import io
import gc
import fitz
import torch
import base64
import transformers
from PIL import Image
import pypdfium2 as pdfium
from dotenv import load_dotenv
from fastembed import SparseTextEmbedding

from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage

c:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
transformers.utils.logging.set_verbosity_info()
enable_progress_bars()

In [3]:
load_dotenv()

True

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [44]:
pdf_path = "pdfs\smartnation2-report.pdf"

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_14292\1914289470.py:1: SyntaxWarning: invalid escape sequence '\s'
  pdf_path = "pdfs\smartnation2-report.pdf"


In [45]:
pdf = pdfium.PdfDocument(pdf_path)
total_pages = len(pdf)

In [46]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.mode = "accurate"
pipeline_options.accelerator_options = AcceleratorOptions(
    device = AcceleratorDevice.CUDA
)

In [47]:
md_index = {}

In [48]:
def sanitize_string(s: str) -> str:
    if s is None:
        return None
    return s.replace('\x00', '')

In [49]:
batch_size = 30

In [50]:
for start in range(1, total_pages + 1, batch_size):
    end = min(start + batch_size - 1, total_pages)
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backened=PyPdfiumDocumentBackend
            )
        }
    )

    result = converter.convert(pdf_path, page_range=(start, end))
    doc = result.document
    for page_num in doc.pages.keys():
        markdown_content = doc.export_to_markdown(page_no=page_num)
        md_index[page_num] = sanitize_string(markdown_content)

    del converter
    gc.collect()
    torch.cuda.empty_cache()
    



[INFO] 2026-05-06 15:18:41,459 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-06 15:18:41,500 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-06 15:18:41,500 [RapidOCR] main.py:57: Using C:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-06 15:18:41,691 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-06 15:18:41,708 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-06 15:18:41,709 [RapidOCR] main.py:57: Using C:\Users\UserAdmin\Documents\RAG\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-06 15:18:41,766 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-06 15:18:4

In [55]:
md_index

{1: '## Smart Nation 2.0\n\nA Thriving Digital Future for All\n\n<!-- image -->\n\n<!-- image -->\n\nMINISTRYOF DIGITALDEVELOPMENT ANDINFORMATION',
 2: 'Published by the Ministry of Digital Development and Information, October 2024. We thank all the organisations and individuals that have contributed to this report.',
 3: "## Contents\n\n02\n\nForeword by Prime Minister\n\n04\n\nIntroduction\n\n28\n\nCHAPTER 2\n\nTrust\n\n82\n\nCHAPTER 5 Harnessing Technology for Global Good\n\n03\n\nForeword by Minister\n\n10\n\nCHAPTER 1 Improving Citizens' Lives\n\n46\n\nCHAPTER 3 Growth\n\n89\n\nConclusion\n\n66\n\nCHAPTER 4 Community\n\n92\n\nFrameworks and Blueprints",
 4: "## Foreword by Prime Minister\n\nI n 2014, Singapore set an ambition to become a Smart Nation. It was anchored on a vision to use technology to improve our citizens' lives.\n\nTen years on, we have achieved many of our original ambitions. We have developed digital products and services that made everyday life more convenient a

In [56]:
fitz_doc = fitz.open(pdf_path)

In [57]:
enhanced_md_index = md_index.copy()

In [58]:
for page_num, content in md_index.items():
    if "<!-- image -->" in content:
        page = fitz_doc[page_num-1]
        matrix = fitz.Matrix(2, 2) 
        pix = page.get_pixmap(matrix=matrix)
        img_bytes = pix.tobytes("png")
        base64_image = base64.b64encode(img_bytes).decode('utf-8')
        image_data_url = f"data:image/png;base64,{base64_image}"

        prompt = '''
        You are a document processing specialist. You are provided with:
        1. page image
        2. page markdown that contains all extracted text and tables
        
        The page markdown contains image placeholders in this format: <!-- image -->

        Instructions:
        1. DECORATOR IMAGES
        If the image is decorative (background shapes, dividers, logos, watermarks, 
        borders, icons with no informational content) replace with: [DECORATIVE IMAGE]
        
        2. EXTRACT ALL ENTITIES
        List every named element explicitly: people, organisations, systems, values, percentages, dollar amounts, step names, category names AS IT IS. 
        Never say "various items" or "several steps" — name them all AS IT IS. 

        3. EXTRACT RELATIONSHIPS
        Identify the visual type first, then apply the corresponding relationship rule:
        - FLOWCHART: sequence of steps, triggers, decision points, branches
        - VENN: what each circle means, what overlaps imply
        - TABLE: what each row/column combination means as a sentence
        - PIE/BAR/LINE CHART: relative magnitudes, trends, outliers, implications  
        - HEATMAP: axes, colour scale meaning, highest/lowest cells with exact values, visible patterns and what they suggest
        - NETWORK: nodes, edge direction, hub nodes
        - SCREENSHOT/UI: every button, field, menu, navigation element and its purpose

        4. USE QUERY LANGUAGE
        Write as if answering questions a user might ask about this diagram.
        Include synonyms and alternative phrasings.

        5. MAKE IMPLICIT EXPLICIT  
        State conclusions the diagram implies but does not say.
        Example: if construction is 60% → "construction represents the majority 
        of procurement opportunities by value, making it the largest segment 
        for potential suppliers"

        6. FORMAT
        Keep all other markdown exactly as is
        Return only the enriched markdown, no preamble
        '''

        messages = [
            SystemMessage(content = prompt),
            HumanMessage([
                {"type": "text", "text": content},
                {"type": "image_url", "image_url": {"url": image_data_url}}

        ])
        ]

        response = llm.invoke(messages)
        enhanced_markdown = response.content
        enhanced_md_index[page_num] = enhanced_markdown
        print(f"{page_num} modified")
        
        



1 modified
4 modified
MuPDF error: format error: No common ancestor in structure tree

5 modified
6 modified
7 modified
8 modified
9 modified
10 modified
11 modified
12 modified
13 modified
14 modified
15 modified
MuPDF error: format error: No common ancestor in structure tree

16 modified
17 modified
18 modified
MuPDF error: format error: No common ancestor in structure tree

19 modified
20 modified
21 modified
MuPDF error: format error: No common ancestor in structure tree

22 modified
23 modified
MuPDF error: format error: No common ancestor in structure tree

24 modified
25 modified
26 modified
27 modified
28 modified
29 modified
30 modified
31 modified
32 modified
33 modified
34 modified
35 modified
36 modified
37 modified
38 modified
39 modified
40 modified
41 modified
42 modified
43 modified
45 modified
47 modified
48 modified
49 modified
50 modified
51 modified
52 modified
53 modified
54 modified
55 modified
56 modified
57 modified
58 modified
59 modified
60 modified
61 modifie

In [59]:
enhanced_md_index

{1: '## Smart Nation 2.0\n\nA Thriving Digital Future for All\n\n[DECORATIVE IMAGE]\n\n[DECORATIVE IMAGE]\n\n**Entities:**\n- Smart Nation 2.0\n- Ministry of Digital Development and Information (MDDI)\n- Singapore\n\n**Relationships:**\n- The title "Smart Nation 2.0" suggests a new phase or initiative aimed at enhancing digital integration and accessibility for all citizens.\n- The phrase "A Thriving Digital Future for All" implies inclusivity and a focus on community engagement in the digital landscape.\n- The presence of the Ministry of Digital Development and Information indicates government involvement in promoting and facilitating this initiative.\n\n**Conclusions:**\n- The initiative aims to foster a digitally inclusive environment, ensuring that all segments of society can benefit from technological advancements.\n- The visual representation of diverse individuals engaging with technology suggests a community-oriented approach to digital literacy and access.',
 2: 'Published by 

In [32]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


loading configuration file config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-base\snapshots\9fe8a713422a7cb4ef79ca77a09b381ee2243101\config.json
Model config Qwen2VLConfig {
  "architectures": [
    "ColQwen2"
  ],
  "dtype": "float32",
  "image_token_id": 151655,
  "model_type": "qwen2_vl",
  "text_config": {
    "attention_dropout": 0.0,
    "bos_token_id": 151643,
    "dtype": "float32",
    "eos_token_id": 151645,
    "hidden_act": "silu",
    "hidden_size": 1536,
    "initializer_range": 0.02,
    "intermediate_size": 8960,
    "layer_types": [
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_attention",
      "full_atten

In [33]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

loading configuration file processor_config.json from cache at None
loading configuration file preprocessor_config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-v1.0\snapshots\83a0134c8f274b3688d8dbde26de8a5b109ad8b4\preprocessor_config.json
loading configuration file preprocessor_config.json from cache at C:\Users\UserAdmin\.cache\huggingface\hub\models--vidore--colqwen2-v1.0\snapshots\83a0134c8f274b3688d8dbde26de8a5b109ad8b4\preprocessor_config.json
Image processor Qwen2VLImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "merge_size": 2,
  "patch_size": 14,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 602112,
    "shortest_edge": 3136
  },
  "temporal_pa

In [60]:
vector_index = []

In [61]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [62]:
for i in range(len(fitz_doc)):
    page = fitz_doc.load_page(i)
    pix = page.get_pixmap(dpi=150)
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    
    inputs = processor.process_images(images=[img])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings)
    
    vector_index.append({
        "page_number": i+1,
        "coarse_vector": coarse_vector.cpu(),
        "page_embeddings": embeddings.cpu()
        
    })


MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree



docker exec -it pgvector-db psql -U postgres

In [63]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)

In [64]:
try:     
    with conn.cursor() as cur:
        for item in vector_index:
            page_number = item["page_number"]
            page_md = md_index.get(page_number, "")

            cur.execute(
                '''
                INSERT INTO pages(page_number, coarse_vector, markdown)
                VALUES (%s, %s, %s) RETURNING id;
                ''',
                (page_number, item["coarse_vector"].flatten().tolist(), page_md)
            )

            page_id = cur.fetchone()[0]

            patch_vectors = item["page_embeddings"].squeeze(0).to(torch.float32).cpu().numpy()

            patch_rows = []
            for index, vec in enumerate(patch_vectors):
                patch_rows.append((
                    page_id, 
                    index, 
                    vec.tolist()
                ))

            cur.executemany(
                '''
                INSERT INTO page_patches (page_id, patch_index, patch_embedding)
                VALUES (%s, %s, %s);
                ''',
                patch_rows
            )

        conn.commit()

except Exception as e: 
        print(f"Error: {e}")
        conn.rollback()

In [64]:
import re
import numpy as np
from collections import Counter

def idf(md_index):
    total_pages =len(md_index)
    doc_frequency = Counter()

    for _, text in md_index.items():
        tokens = set(re.findall(r'[a-z0-9\$]+', text.lower()))
        for token in tokens:
            doc_frequency[token] += 1
    idf_map = {token: np.log(total_pages / count) for token, count in doc_frequency.items()}
    return idf_map

In [65]:
idf_map = idf(md_index)

In [66]:
import json
with open("idf_map.json", "w") as f:
    json.dump(idf_map, f)

In [40]:
enhanced_md_index

{1: '<!-- image -->\n\n## Login User Guide\n\nA guide to log into Vendors@Gov portal.\n\n<!-- image -->\n\n[DECORATIVE IMAGE]\n\n### Extracted Entities\n- **Portal**: Vendors@Gov\n- **Document Title**: Login User Guide\n\n### Implicit Conclusions\nThe document serves as a user guide specifically designed to assist users in logging into the Vendors@Gov portal.',
 2: '## Vendors@Gov Log In User Guide\n\n1. **For UEN Registered Company/ Organisation**  \n   This login method is applicable to UEN registered vendors who are transacting as a Company/ Organisation, Sole Proprietors, or Societies.\n\n2. **For Foreign Company/ Organisation without UEN**  \n   This login method is applicable to non-UEN registered vendors who are transacting as a Foreign Company/ Organisation.\n\n3. **For Individuals with Singpass**  \n   This login method is applicable to vendors who are transacting as an individual for your personal payment matters (e.g. freelancers).\n\n4. **For Foreign Individuals with AGD Pa